# RAG (Embedding Search) — `career_position` Annotation — Broad sectors

For each test entry, retrieves the top-5 most similar career entries from the
training set **within the same country**, shows their assigned broad sector codes
as labelled examples in the prompt, then asks the LLM to classify.

**Classification model:** TBD — set `MODEL_ID` below  
**Embedding model:** LM Studio local — set `EMBEDDING_MODEL_ID` below  
**Requires:** `OPENAI_API_KEY` (or LM Studio) in `.env`

## 1. Imports & config

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
import re
import time
from pathlib import Path

import anthropic
import numpy as np
import pandas as pd
from openai import OpenAI
from tqdm import tqdm

from corex_eval import (
    evaluate, load_inputs, load_training_data,
    career_position_to_sector, CAREER_POSITION_SECTORS, CAREER_POSITION_SECTOR_HINTS,
)

MODEL_ID           = "claude-opus-4-6"
TEMPERATURE        = 0
MAX_TOKENS         = 16

EMBEDDING_MODEL_ID = "BAAI/bge-m3"
LMSTUDIO_URL       = "http://localhost:1234/v1"
EMBED_BATCH_SIZE   = 32
K                  = 5

CACHE_DIR          = Path("cache")
CACHE_DIR.mkdir(exist_ok=True)
EMBED_CACHE_FILE   = CACHE_DIR / "train_embeddings.npy"

client       = anthropic.Anthropic()
embed_client = OpenAI(base_url=LMSTUDIO_URL, api_key="lm-studio")
print(f"Classification model : {MODEL_ID}")
print(f"Embedding model      : {EMBEDDING_MODEL_ID}")

## 2. Load training data (collapse to broad sectors)

In [ ]:
train_df = load_training_data(
    task="annotation",
    variable="career_position",
    features=["job_description_label", "workplace_label", "career_position", "country_label"],
)
train_df = train_df.dropna(subset=["job_description_label", "career_position"]).reset_index(drop=True)
train_df["sector"]     = train_df["career_position"].map(career_position_to_sector)
train_df["input_text"] = train_df["job_description_label"] + " | " + train_df["workplace_label"].fillna("")
print(f"{len(train_df)} training rows across {train_df['country_label'].nunique()} countries")

## 3. Compute / load training embeddings (cached)

In [ ]:
def batch_embed(texts):
    all_embeddings = []
    for i in tqdm(range(0, len(texts), EMBED_BATCH_SIZE), desc="Embedding"):
        batch = texts[i : i + EMBED_BATCH_SIZE]
        resp  = embed_client.embeddings.create(input=batch, model=EMBEDDING_MODEL_ID)
        all_embeddings.extend([e.embedding for e in resp.data])
    return np.array(all_embeddings, dtype=np.float32)

def single_embed(text):
    resp = embed_client.embeddings.create(input=[text], model=EMBEDDING_MODEL_ID)
    return np.array(resp.data[0].embedding, dtype=np.float32)

if EMBED_CACHE_FILE.exists():
    train_embeddings = np.load(EMBED_CACHE_FILE)
    print(f"Loaded {len(train_embeddings)} cached embeddings")
else:
    train_embeddings = batch_embed(train_df["input_text"].tolist())
    np.save(EMBED_CACHE_FILE, train_embeddings)
    print(f"Computed and cached {len(train_embeddings)} embeddings")

## 4. Build system prompt

In [ ]:
sectors_text = "\n".join(
    f"  {k} = {v}: {CAREER_POSITION_SECTOR_HINTS[k]}"
    for k, v in sorted(CAREER_POSITION_SECTORS.items())
)

SYSTEM_PROMPT = (
    "You are an expert political scientist coding political career histories "
    "using the CoREx codebook.\n\n"
    "Task: given a career entry (job title and workplace), assign the single most "
    "appropriate broad sector code. You will also be shown similar entries from the "
    "training data and their assigned sectors as a reference.\n"
    'Respond with ONLY the sector number (e.g. "1") — nothing else.\n\n'
    f"Broad sector codes:\n{sectors_text}"
)

valid_codes = [str(k) for k in sorted(CAREER_POSITION_SECTORS.keys(), key=int)]
print(f"Broad sectors: {valid_codes}")

## 5. Load test inputs

In [ ]:
test_df = load_inputs(task="annotation", variable="career_position")
print(f"{len(test_df)} test rows")
test_df[["case_id", "spell_index", "job_description_label", "workplace_label", "country_label"]].head()

## 6. Retrieval helper

In [ ]:
def find_top_k(query_emb, country, k=K):
    mask = train_df["country_label"].astype(str) == str(country)
    if mask.sum() == 0:
        return []
    country_embs = train_embeddings[mask]
    country_rows = train_df[mask].reset_index(drop=True)
    norms = np.linalg.norm(country_embs, axis=1)
    sims  = (country_embs @ query_emb) / (norms * np.linalg.norm(query_emb) + 1e-10)
    top_k = np.argsort(sims)[-k:][::-1]
    return [
        (country_rows.iloc[i]["job_description_label"],
         country_rows.iloc[i].get("workplace_label", ""),
         country_rows.iloc[i]["sector"])
        for i in top_k
    ]

def format_examples(examples):
    if not examples:
        return ""
    lines = ["\nSimilar entries from the training data:"]
    for i, (title, wp, sector) in enumerate(examples, 1):
        sector_label = CAREER_POSITION_SECTORS.get(sector, sector)
        lines.append(f'  {i}. "{title}" | "{wp}" → {sector} = {sector_label}')
    lines.append("\nNow classify:")
    return "\n".join(lines)

## 7. Inference

In [ ]:
def parse_prediction(text, valid_codes):
    text = text.strip()
    if text in valid_codes:
        return text
    for code in valid_codes:
        if text.startswith(code):
            return code
    return text


predictions = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Inference"):
    job_title = row["job_description_label"]
    workplace = row.get("workplace_label", "") or ""
    country   = row.get("country_label", "")

    query_emb = single_embed(f"{job_title} | {workplace}")
    examples  = find_top_k(query_emb, country)
    ex_block  = format_examples(examples)

    user_msg = f"Job title: {job_title}\nWorkplace: {workplace or 'unknown'}{ex_block}"

    try:
        resp = client.messages.create(
            model=MODEL_ID,
            system=SYSTEM_PROMPT,
            messages=[{"role": "user", "content": user_msg}],
            temperature=TEMPERATURE,
            max_tokens=MAX_TOKENS,
        )
        raw = resp.content[0].text
    except Exception as e:
        print(f"Error on {row['case_id']}/{row['spell_index']}: {e}")
        raw = ""
        time.sleep(0.5)

    predictions.append(parse_prediction(raw, valid_codes))

predictions_df = test_df[["case_id", "spell_index"]].copy()
predictions_df["predicted_code"] = predictions
predictions_df.head()

## 8. Evaluate

In [ ]:
results = evaluate(
    predictions_df,
    task="annotation",
    variable="career_position",
    granularity="broad",
)
pd.Series({k: v for k, v in results.items() if k not in ("per_class", "per_country")})